# FrozenSE — nnU-Net baseline (v1.6.2)

Runs **nnU-Net v2 (2D)** on exactly the same six leave-one-slide-out folds and
the same patches as every other model in the study: the 1,225 training patches
of each fold, its validation slide, and its test slide. nnU-Net plans,
preprocesses and trains itself; the only fixed choices are the training length
(`NNUNET_EPOCHS`), no test-time augmentation (as for every other model), and
the train/validation split, which is ours so the test slide is never seen.

**Inputs:** the same three datasets as the main notebook (`gleason2019`,
`mobile_sam.pt`, the `frozense` package). From the second session on, also attach
this notebook's previous output, so finished folds are skipped.

Run with *Save Version → Save & Run All*, accelerator **GPU T4 × 2**, internet on.
Each fold takes about 2 to 3.5 hours; the two GPUs run folds in parallel, and a
fold is never started unless it can finish before the 12-hour limit.


In [ ]:
# ===================== CHOOSE WHAT THIS RUN DOES =====================
STAGE = 'report'   # fixed: this notebook only runs nnU-Net
# =====================================================================

import os, sys, time, glob, json, shutil, zipfile, importlib, importlib.util
SESSION_START = time.time()
TRAINING_STAGES = {'smoke', 'baselines', 'improve', 'ablations', 'data_efficiency', 'extra'}
assert STAGE in TRAINING_STAGES | {'setup', 'report'}, f"unknown STAGE {STAGE!r}"

ON_KAGGLE = os.path.isdir('/kaggle/input')
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
# Kaggle's newer images ship the google.colab package too, so importing it
# proves nothing there: Kaggle wins.
ON_COLAB = ON_COLAB and not ON_KAGGLE

if ON_KAGGLE:
    PLATFORM    = 'kaggle'
    SEARCH_ROOT = '/kaggle/input'                  # your datasets + earlier outputs
    SCRATCH     = '/kaggle/tmp'                    # working space, not saved
    OUTPUT_ROOT = '/kaggle/working/frozense_runs'  # saved with each version
    # A Kaggle session ends at 12 hours, and output from a session killed at
    # the limit may never be saved. So no fold is STARTED unless it is
    # expected to finish within 11.5 hours; the run then ends cleanly and
    # the next session continues.
    DEADLINE = SESSION_START + 11.5 * 3600
elif ON_COLAB:
    PLATFORM    = 'colab'
    SEARCH_ROOT = '/content/drive/MyDrive/frozense'
    SCRATCH     = '/content'
    OUTPUT_ROOT = '/content/drive/MyDrive/frozense_runs'
    DEADLINE    = None   # each fold is written to Drive the moment it ends
else:
    PLATFORM    = 'local'
    SEARCH_ROOT = os.path.abspath('frozense_inputs')
    SCRATCH     = os.path.abspath('frozense_scratch')
    OUTPUT_ROOT = os.path.abspath('frozense_runs')
    DEADLINE    = None

os.makedirs(SCRATCH, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"platform: {PLATFORM}   STAGE: {STAGE!r}")
print(f"inputs  : {SEARCH_ROOT}")
print(f"outputs : {OUTPUT_ROOT}")

## 1 · Environment

Detects Kaggle or Colab, sets the input and output folders and records the session start time.


In [ ]:
!pip install -q "segmentation_models_pytorch==0.6.0" "albumentations==2.0.8" "timm>=1.0" git+https://github.com/ChaoningZhang/MobileSAM.git

In [ ]:
missing = [m for m in ('segmentation_models_pytorch', 'albumentations', 'timm', 'mobile_sam')
           if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(
        f"Not installed: {missing}. On Kaggle, turn on Settings -> Internet "
        "(it needs a verified phone number), then run the cell above again.")

import torch, numpy as np
import segmentation_models_pytorch as smp, albumentations as A, timm
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | "
      f"smp {smp.__version__} | albumentations {A.__version__} | timm {timm.__version__}")

# Check the GPU with a REAL operation, not just by asking whether one exists.
# Kaggle's P100 currently passes the existence check but fails on its first
# computation with "no kernel image is available", because the PyTorch build
# Kaggle ships no longer supports that GPU generation.
GPU_OK, gpu_problem = False, None
if torch.cuda.is_available():
    try:
        x = torch.randn(2, 3, 64, 64, device='cuda', requires_grad=True)
        w = torch.randn(4, 3, 3, 3, device='cuda')
        torch.nn.functional.conv2d(x, w).sum().backward()
        torch.cuda.synchronize()
        GPU_OK = True
        print("GPU:", torch.cuda.get_device_name(0))
    except Exception as exc:
        gpu_problem = f"{type(exc).__name__}: {exc}"

DEVICE = torch.device('cuda' if GPU_OK else 'cpu')
NUM_WORKERS = min(4, os.cpu_count() or 2)

if STAGE in TRAINING_STAGES and not GPU_OK:
    raise RuntimeError(
        f"STAGE {STAGE!r} trains models and needs a working GPU. "
        + (f"The GPU failed a test operation ({gpu_problem}). " if gpu_problem else
           "No GPU is attached. ")
        + "On Kaggle: Settings -> Accelerator -> GPU T4 x2 (not P100). "
          "On Colab: Runtime -> Change runtime type -> GPU.")
if not GPU_OK:
    print(f"No GPU in use -- fine for STAGE {STAGE!r}.")

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

### Install the `frozense` package

Finds the `frozense` package among the attached inputs (a folder or a zip), copies it to scratch space and imports it. The printed version should be 1.6.2.


In [ ]:
PKG_ROOT = os.path.join(SCRATCH, 'frozense_pkg')


def safe_extract(zip_path, dest):
    # Extract a zip, accepting EITHER path separator.
    #
    # Before version 1.2.3 of its Archive module, Windows PowerShell's
    # Compress-Archive wrote entries with backslashes
    # ('imagesTr\slide001_core004.jpg'). On Linux, Python treats a backslash
    # as an ordinary character, so a plain extractall() leaves every file
    # sitting flat in one folder with backslashes in its NAME -- and every
    # later search for imagesTr/*.jpg finds nothing.
    dest_real = os.path.realpath(dest)
    n = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/').lstrip('/')
            if not name or name.endswith('/'):
                continue                      # directory entry
            target = os.path.realpath(os.path.join(dest, name))
            if not target.startswith(dest_real + os.sep):
                raise ValueError(f"refusing to extract outside {dest}: {info.filename}")
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as out:
                shutil.copyfileobj(src, out, 1 << 20)
            n += 1
    return n


def find_files(pattern):
    return sorted(glob.glob(f'{SEARCH_ROOT}/**/{pattern}', recursive=True),
                  key=os.path.getmtime, reverse=True)


if not os.path.isdir(SEARCH_ROOT):
    raise FileNotFoundError(
        f"{SEARCH_ROOT} does not exist. " +
        ("Create a folder named 'frozense' directly inside My Drive and put "
         "gleason2019.zip, mobile_sam.pt and FrozenSE_SAM_Revision_Package.zip in it."
         if ON_COLAB else "Add your datasets to the notebook (right panel -> Add Input)."))

pkg_zips = find_files('FrozenSE_SAM_Revision_Package*.zip')
pkg_dirs = sorted(os.path.dirname(p) for p in
                  glob.glob(f'{SEARCH_ROOT}/**/frozense/__init__.py', recursive=True))

shutil.rmtree(PKG_ROOT, ignore_errors=True)
if pkg_zips:
    PKG_SOURCE = pkg_zips[0]
    tmp = os.path.join(SCRATCH, '_pkg_unzip')
    shutil.rmtree(tmp, ignore_errors=True)
    safe_extract(PKG_SOURCE, tmp)
    inner = glob.glob(f'{tmp}/**/frozense/__init__.py', recursive=True)
    if not inner:
        raise FileNotFoundError(f"{PKG_SOURCE} contains no frozense/ package")
    shutil.copytree(os.path.dirname(inner[0]), f'{PKG_ROOT}/frozense')
    shutil.rmtree(tmp, ignore_errors=True)
elif pkg_dirs:
    PKG_SOURCE = pkg_dirs[0]
    shutil.copytree(PKG_SOURCE, f'{PKG_ROOT}/frozense')
else:
    raise FileNotFoundError(
        f"No frozense package anywhere under {SEARCH_ROOT}. Upload "
        "FrozenSE_SAM_Revision_Package.zip " +
        ("into that Drive folder." if ON_COLAB else "as a Kaggle dataset and add it as an input."))

# After a FAILED import, Python remembers 'nothing here' for this path, so
# copying the package in is not enough on its own; after an earlier
# SUCCESSFUL one, old modules stay loaded. Clear both.
for name in [m for m in sys.modules if m == 'frozense' or m.startswith('frozense.')]:
    del sys.modules[name]
sys.path_importer_cache.pop(PKG_ROOT, None)
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)
importlib.invalidate_caches()

import frozense
print(f"frozense {frozense.__version__}")
print(f"  source : {PKG_SOURCE}")
print(f"  loaded : {os.path.dirname(frozense.__file__)}")

In [ ]:
# Results from earlier sessions. On Colab they are already in Drive. On
# Kaggle they arrive as the attached output of your previous version (right
# panel -> Add Input -> Your Work -> this notebook), packaged as
# frozense_runs.zip; they are unpacked into this session's output so that
# finished folds are loaded instead of retrained.
if ON_KAGGLE:
    for z in sorted(find_files('*frozense[-_]runs*.zip'), key=os.path.getmtime):
        safe_extract(z, OUTPUT_ROOT)
    for d in glob.glob(f'{SEARCH_ROOT}/**/*frozense[-_]runs*', recursive=True):
        if os.path.isdir(d):
            shutil.copytree(d, OUTPUT_ROOT, dirs_exist_ok=True)

saved = glob.glob(f'{OUTPUT_ROOT}/**/fold_result.json', recursive=True)
print(f"{len(saved)} saved fold results available from earlier sessions")

## 2 · Data

Inputs the notebook expects to find among the attached datasets:

- the Gleason 2019 folders `imagesTr`, `imagesVal`, `Generated_Labels_Train`,
  `labelsVal` and `annotations` (one subfolder per pathologist, `Map1` to
  `Map6`), unpacked or as one zip;
- `mobile_sam.pt`, the official MobileSAM weights;
- the `frozense` package (folder or zip).

On Colab, put the same files in `MyDrive/frozense/`. The cell below handles
data that is unpacked, zipped, or unpacked flat with backslashes in the file
names (a Windows-made zip opened on Linux). Expected: 55 + 138 = **193
labelled cores**.


In [ ]:
FOLDERS = ('imagesTr', 'imagesVal', 'Generated_Labels_Train', 'labelsVal', 'annotations')
RAW = os.path.join(SCRATCH, 'raw')
MARK = os.path.join(RAW, '.prepared_from')

weights = find_files('mobile_sam*.pt')
if not weights:
    raise FileNotFoundError(f"No mobile_sam.pt anywhere under {SEARCH_ROOT}")
WEIGHTS = weights[0]


def locate(root):
    out = {}
    for name in FOLDERS:
        hits = sorted(d for d in glob.glob(f'{root}/**/{name}', recursive=True)
                      if os.path.isdir(d))
        if hits:
            out[name] = hits[0]
    return out


def flat_backslash_files(root):
    found = []
    for r, _, files in os.walk(root):
        found += [os.path.join(r, f) for f in files
                  if '\\' in f and f.split('\\')[0] in FOLDERS]
    return found


DATA, how = {}, None
if ON_KAGGLE:
    in_place = locate(SEARCH_ROOT)
    if len(in_place) == len(FOLDERS):
        DATA, how = in_place, 'read in place from your Kaggle dataset'

if not DATA:
    flat = flat_backslash_files(SEARCH_ROOT)
    zips = find_files('gleason2019*.zip')
    source = (f'flat:{os.path.dirname(flat[0])}' if flat else
              f'zip:{zips[0]}' if zips else None)
    if source is None:
        raise FileNotFoundError(
            f"Found neither the labelled folders, nor gleason2019.zip, under "
            f"{SEARCH_ROOT}. Check the inputs are attached.")

    done = os.path.exists(MARK) and open(MARK).read() == source
    if not done:
        # The marker is written only AFTER a complete job, so a session that
        # dropped mid-way is redone rather than mistaken for finished.
        shutil.rmtree(RAW, ignore_errors=True)
        os.makedirs(RAW)
        if flat:
            print(f"rebuilding folders for {len(flat)} files unpacked flat ...")
            for f in flat:
                rel = os.path.basename(f).split('\\')
                dest = os.path.join(RAW, *rel)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy(f, dest)
        else:
            print("copying the dataset zip to local disk (a few minutes) ...")
            local = os.path.join(SCRATCH, 'gleason.zip')
            shutil.copy(zips[0], local)
            print("extracting ...")
            safe_extract(local, RAW)
            os.remove(local)
        with open(MARK, 'w') as fh:
            fh.write(source)
    DATA, how = locate(RAW), ('rebuilt from flat files' if flat else 'extracted from the zip')

absent = [f for f in FOLDERS if f not in DATA]
if absent:
    raise FileNotFoundError(f"Missing folders: {absent}. Found: {DATA}")

print(f"data    : {how}")
print(f"weights : {WEIGHTS}\n")
for name in FOLDERS:
    if name == 'annotations':
        maps = sorted(d for d in os.listdir(DATA[name]) if os.path.isdir(os.path.join(DATA[name], d)))
        print(f"  {name:<24}{len(maps):>5} reader folders ({', '.join(maps)})")
    else:
        print(f"  {name:<24}{len(os.listdir(DATA[name])):>5} files")

### Index every core where it lies

Builds one index of all labelled cores, with the slide each belongs to. Expected: 193 cores on 6 slides.


In [ ]:
IMAGE_DIRS = [DATA['imagesTr'], DATA['imagesVal']]
LABEL_DIRS = [DATA['Generated_Labels_Train'], DATA['labelsVal']]
ANN_DIR = DATA['annotations']

SOURCE_OF, LBL_SOURCE, clashes = {}, {}, []


def register(folder_name, registry, exts, strip=''):
    for f in sorted(os.listdir(DATA[folder_name])):
        stem, ext = os.path.splitext(f)
        if ext.lower() not in exts:
            continue
        core_id = stem[:-len(strip)] if strip and stem.endswith(strip) else stem
        if core_id in registry:
            clashes.append((core_id, registry[core_id], folder_name))
        registry[core_id] = folder_name


for name in ('imagesTr', 'imagesVal'):
    register(name, SOURCE_OF, {'.jpg', '.jpeg', '.png'})
for name in ('Generated_Labels_Train', 'labelsVal'):
    register(name, LBL_SOURCE, {'.png'}, strip='_classimg_nonconvex')

print(f"images: {len(SOURCE_OF)} cores   labels: {len(LBL_SOURCE)} cores")
if clashes:
    # The same core in two source folders means they overlap, which would be
    # worth knowing before trusting any of the original numbers.
    print(f"WARNING: {len(clashes)} core(s) appear in two source folders, "
          f"e.g. {clashes[:3]}")

### 2b · Audit the labels before trusting anything

Checks label values and foreground fractions on a sample of cores and reports cores with no tumour pixels.


In [ ]:
if PKG_ROOT not in sys.path:   # installed in step 1
    sys.path.insert(0, PKG_ROOT)
from frozense.config import *
from frozense.data import (
    discover_cores, make_split, n_folds, PatchDataset,
    audit_label_values, format_label_audit, group_composition,
)

base = ExperimentConfig(
    name='proposed_frozense_sam',
    split=SplitConfig(
        group_regex=r'^(?P<group>slide\d+)',   # matches slide001_core004.jpg
        allow_core_level_fallback=False,
        scheme='leave_one_group_out',
        n_train_cores=35, seed=42,
    ),
    data=DataConfig(
        images_dir=IMAGE_DIRS, labels_dir=LABEL_DIRS,   # read in place, both folders
        # slide001_core004.jpg  ->  slide001_core004_classimg_nonconvex.png
        label_suffixes=('_classimg_nonconvex', ''),
        tumour_grades=(3, 4, 5),               # <-- the audit below checks this
    ),
    model=ModelConfig(sam_checkpoint=WEIGHTS),
    output_root=OUTPUT_ROOT,
)
# Training budget. 30 epochs with patience 10 fits the free Kaggle quota;
# every model gets the same budget, so comparisons stay fair. State it in
# the methods. (The library default is 300 / 40.)
base = replace_nested(base, **{'train.num_workers': NUM_WORKERS,
                               'train.epochs': 30, 'train.early_stopping_patience': 10})

# The final model, chosen on validation in step 5b. Ablations and the
# learning curve are run on THIS design, not the original.
from frozense.experiments import IMPROVED_PROTOCOL
from frozense.experiments import build_improvements as _bi
UNET_R2 = next(e.cfg for e in _bi(base) if e.cfg.name == 'baseline_unet_pretrained_r2')
FINAL = replace_nested(base, name='improved_skip_partial',
                       **{'model.use_skip_connections': True,
                          'model.unfreeze_last_n_blocks': 1}, **IMPROVED_PROTOCOL)

cores = discover_cores(base.data, base.split)
print(f"{len(cores)} cores, {n_folds(cores)} slides\n")

comp = group_composition(cores, SOURCE_OF)
for slide, info in sorted(comp['per_group'].items()):
    src = ', '.join(f"{k} {v}" for k, v in sorted(info['sources'].items()))
    print(f"  {slide}: {info['n_cores']:>3} cores   ({src})")

shared = comp['groups_spanning_multiple_sources']
print(f"\nslides present in BOTH original source folders: {shared or 'none'}")
if shared:
    print("  -> the released imagesTr/imagesVal partition is NOT slide-disjoint.")
    print("     Same staining batch in training and in the set used for both")
    print("     checkpoint selection and final reporting.")

## nnU-Net


In [ ]:
!pip install -q "nnunetv2==2.6.4"

In [ ]:
import subprocess, torch
from dataclasses import asdict

NNUNET_EPOCHS = 50          # nnUNetTrainer_50epochs: 50 x 250 iterations
NN_ROOT = os.path.join(SCRATCH, 'nnunet')     # rebuilt every session, not saved
for d in ('raw', 'preprocessed', 'results'):
    os.makedirs(os.path.join(NN_ROOT, d), exist_ok=True)

job = {'base': asdict(base), 'cores': [asdict(c) for c in cores],
       'folds': list(range(n_folds(cores))), 'output_root': OUTPUT_ROOT,
       'raw': f'{NN_ROOT}/raw', 'preprocessed': f'{NN_ROOT}/preprocessed',
       'results': f'{NN_ROOT}/results', 'scratch': NN_ROOT,
       'dataset_base_id': 500, 'epochs': NNUNET_EPOCHS, 'deadline': DEADLINE,
       'fold_hours_estimate': 3.5, 'n_proc_da': 2, 'n_proc_pre': 2}
job_path = os.path.join(SCRATCH, 'nnunet_job.json')
json.dump(job, open(job_path, 'w'), default=str)

done = [f for f in job['folds']
        if os.path.exists(f'{OUTPUT_ROOT}/nnunet_2d/fold{f}/result.json')]
print(f"folds already finished: {done or 'none'}")

N = max(1, min(torch.cuda.device_count(), 2))
logs = [os.path.join(SCRATCH, f'nnunet_worker{i}.log') for i in range(N)]
procs = []
for i in range(N):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i),
               PYTHONPATH=os.pathsep.join(p for p in (PKG_ROOT, os.environ.get('PYTHONPATH')) if p))
    procs.append(subprocess.Popen(
        [sys.executable, '-u', '-m', 'frozense.nnunet_bridge', '--job', job_path,
         '--worker', str(i), '--n-workers', str(N)],
        stdout=open(logs[i], 'w'), stderr=subprocess.STDOUT, env=env, cwd=SCRATCH))
offsets = [0] * N
while any(p.poll() is None for p in procs):
    time.sleep(30)
    for i, path in enumerate(logs):
        with open(path) as fh:
            fh.seek(offsets[i]); chunk = fh.read(); offsets[i] = fh.tell()
        for line in chunk.splitlines():
            print(f"[gpu {i}] {line}", flush=True)
for i, path in enumerate(logs):
    with open(path) as fh:
        fh.seek(offsets[i]); rest = fh.read()
    for line in rest.splitlines():
        print(f"[gpu {i}] {line}", flush=True)
print("workers finished")

In [ ]:
from frozense.metrics import EvaluationResult
from frozense.crossval import pool_folds
import pandas as pd

res, rows = [], []
for f in range(n_folds(cores)):
    p = f'{OUTPUT_ROOT}/nnunet_2d/fold{f}/result.json'
    if os.path.exists(p):
        r = EvaluationResult.from_dict(json.load(open(p))['result'])
        res.append(r)
        rows.append({'fold': f, 'test_dice': r.mean('dice'), 'cores': len(r.cores)})
print(pd.DataFrame(rows) if rows else "no folds finished yet")
if len(res) == n_folds(cores):
    pooled = pool_folds(res, 'nnunet_2d')
    lo, hi = pooled.bootstrap_ci('dice')
    print(f"\nnnU-Net 2D pooled test Dice over {len(pooled.cores)} cores: "
          f"{pooled.mean('dice'):.4f} [{lo:.4f}, {hi:.4f}]")
    pd.DataFrame([{'core_id': c.core_id, 'dice': c.dice, 'iou': c.iou, 'hd95_mm': c.hd95_mm,
                   'asd_mm': c.asd_mm, 'n_defined_distance': c.n_defined_distance}
                  for c in pooled.cores]).to_csv(f'{OUTPUT_ROOT}/nnunet_per_core.csv', index=False)
    json.dump(pooled.case_counts, open(f'{OUTPUT_ROOT}/nnunet_case_counts.json', 'w'))
    print("all six folds done: send the output zip back.")
else:
    print(f"\n{len(res)} of {n_folds(cores)} folds done: run again with this output attached.")

In [ ]:
if ON_KAGGLE:
    archive = shutil.make_archive('/kaggle/working/frozense_runs', 'zip', OUTPUT_ROOT)
    print(f"packaged {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")
print(f"session time: {(time.time() - SESSION_START) / 3600:.2f} h")